In [0]:
"""
id: orders
template: source
name: orders
position:
  x: 0
  y: 0
description:
  text: Read CSV file from specified path.
  hash: e5e12d0b
previewCodeHash: ad114cca78a1a128
previewMode: "1000"
config:
  file_source:
    path: /Volumes/daphne/tableauprep/raw/orders.csv
    format: '"csv"'
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/daphne/tableauprep/raw/orders.csv",
        "format": "\"csv\""
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["orders.data"] = out["data"]

In [0]:
"""
id: clean_orders
template: sql
name: clean_orders
position:
  x: 260
  y: 0
description:
  text: clean orders
  hash: ""
previewCodeHash: 33d4c795697e71be
previewMode: "1000"
config:
  query: SELECT *, CAST(`placed_at` AS DATE) AS `placed_at`, `amount` AS `revenue` FROM orders WHERE `status` = "paid"
input:
  - node: orders
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *, CAST(`placed_at` AS DATE) AS `placed_at`, `amount` AS `revenue` FROM orders WHERE `status` = \"paid\""
}
inputs = {
    "data": [
        ctx["orders.data"]
    ],
    "data__sources": [
        {
            "node": "orders",
            "output_port": "data",
            "name": "orders",
            "df_name": "orders"
        }
    ]
}
out = run(config, inputs, spark)
ctx["clean_orders.result"] = out["result"]

In [0]:
"""
id: revenue_by_customer
template: aggregate
name: revenue_by_customer
position:
  x: 520
  y: 0
description:
  text: revenue by customer
  hash: ""
previewCodeHash: 598db191b3766319
previewMode: full
config:
  group_bys:
    - expr: customer_id
      type: expr
  aggregations:
    - columnExpr:
        expr: revenue
        type: expr
      fn: SUM
      alias: revenue
    - columnExpr:
        expr: order_id
        type: expr
      fn: COUNT
      alias: order_id
input:
  - node: clean_orders
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        # Skip pass-through columns that duplicate a group-by column;
        # groupBy() already includes them in the output.
        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "customer_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "revenue",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "revenue",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "order_id",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["clean_orders.result"]
}
out = run(config, inputs, spark)
ctx["revenue_by_customer.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: revenue_csv
template: output
name: daphne.tableauprep.simple_output_csv
position:
  x: 780
  y: 0
description:
  text: revenue.csv
  hash: ""
previewMode: "1000"
config:
  catalog: daphne
  schema: tableauprep
  table_name: simple_output_csv
input:
  - node: revenue_by_customer
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphne",
    "schema": "tableauprep",
    "table_name": "simple_output_csv"
}
inputs = {
    "data": ctx["revenue_by_customer.aggregated_data"]
}
out = run(config, inputs, spark)

---
id: pipeline_header
template: markdown
name: simple
position:
  x: 0
  y: -260
dimensions:
  width: 700
  height: 220
config:
  md: |
    # simple

    **Converted from `simple.tflx`** by `convert_tflx.py` (tableau-to-lakeflow-designer skill).

    - 4 nodes in the original Tableau Prep flow
    - Output table: `main.default.flow_output`
---